In [1]:
import json, os, random
import numpy as np
import torch
from pathlib import Path
from datasets import load_from_disk
from huggingface_hub import snapshot_download, login as hf_login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed

CFG_PATH = "configs/gemma.json"  # change if needed
cfg = json.loads(Path(CFG_PATH).read_text())

hf_token = cfg.get("hf_token") or os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
if hf_token:
    hf_login(token=hf_token)

seed = int(cfg["data"].get("seed", 42))
set_seed(seed); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

mcfg = cfg["model"]
model_id = mcfg["model_id"]
tok_id = mcfg.get("tokenizer_id", model_id)

tokenizer = AutoTokenizer.from_pretrained(tok_id, use_fast=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_cfg = None
if mcfg.get("load_in_4bit", False):
    compute_dtype = torch.bfloat16 if mcfg.get("bnb_4bit_compute_dtype", "bfloat16") == "bfloat16" else torch.float16
    quant_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_quant_type=mcfg.get("bnb_4bit_quant_type", "nf4"),
        bnb_4bit_use_double_quant=bool(mcfg.get("bnb_4bit_use_double_quant", True)),
    )

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if cfg["trainer"].get("bf16", False) else None,
    low_cpu_mem_usage=True,
    quantization_config=quant_cfg,
    device_map="auto",
    return_dict=True,
)
model.eval()

/home/diptesh/anaconda3/envs/dial_grpo311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
          (position_embedding): Embedding(4096, 1152)
        )
        (encoder): SiglipEncoder(
          (layers): ModuleList(
            (0-26): 27 x SiglipEncoderLayer(
              (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (self_attn): SiglipAttention(
                (k_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                (v_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                (q_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                (out_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              )
              (layer_norm2): LayerNorm((1152,), eps=

In [2]:
import sys
sys.path.append(str(Path(".").resolve()))

from src.formatting import build_chat_prompt

def build_prompt(tokenizer, system_prompt: str, user_prompt: str, prefer_chat_template: bool = True) -> str:
    if prefer_chat_template and hasattr(tokenizer, "apply_chat_template") and getattr(tokenizer, "chat_template", None):
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": user_prompt})
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return build_chat_prompt(tokenizer, system_prompt, user_prompt)

def load_dataset_from_hub_snapshot(dataset_id: str, split: str = "train"):
    local_path = snapshot_download(dataset_id, repo_type="dataset")
    ds_any = load_from_disk(local_path)
    if hasattr(ds_any, "column_names"):
        return ds_any
    return ds_any[split] if split in ds_any else ds_any[list(ds_any.keys())[0]]

dcfg = cfg["data"]
system_prompt = dcfg.get("system_prompt", "") or ""

ds = load_dataset_from_hub_snapshot(dcfg["dataset_id"], dcfg.get("dataset_split", "train"))
for col in ["prompt", "chosen", "rejected"]:
    assert col in ds.column_names, (col, ds.column_names)

def map_fn(ex):
    raw = ex["prompt"]
    ex["prompt_raw"] = raw
    ex["prompt_chat"] = build_prompt(tokenizer, system_prompt, raw, prefer_chat_template=True)
    return ex

ds = ds.map(map_fn)
print(ds.column_names)
print(ds[0]["prompt_raw"])
print(ds[0]["prompt_chat"][-400:])

Map: 100%|██████████| 45690/45690 [00:06<00:00, 6928.71 examples/s]

['prompt', 'chosen', 'rejected', 'prompt_dialect_density', 'chosen_dialect_density', 'prompt_raw', 'prompt_chat']
a novel plot about whenever your soulmate gets a cut/wound you will grow a flower on the same spot. the story revolves around two boys, one shy and reserved boy full with flowers and the other is a gruff and aggressive boy full of wounds
<bos><start_of_turn>user
You are a helpful assistant.

a novel plot about whenever your soulmate gets a cut/wound you will grow a flower on the same spot. the story revolves around two boys, one shy and reserved boy full with flowers and the other is a gruff and aggressive boy full of wounds<end_of_turn>
<start_of_turn>model



In [ ]:
def hard_trim_completion(text: str, stop_strings):
    if not text:
        return text
    cut = None
    for s in stop_strings:
        if not s:
            continue
        idx = text.find(s)
        if idx != -1:
            cut = idx if cut is None else min(cut, idx)
    return text[:cut].rstrip() if cut is not None else text

stop_strings = [
    "\nUser:", "\nAssistant:",
    "\n### User:", "\n### Assistant:",
    "\n<|user|>", "\n<|assistant|>",
]

@torch.no_grad()
def generate_completion(chat_prompt: str, max_new_tokens=1028, temperature=0.9, top_p=0.95):
    inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    gen_ids = out[0, input_len:]
    completion = tokenizer.decode(gen_ids, skip_special_tokens=True)
    completion = hard_trim_completion(completion, stop_strings)
    return completion.strip()

X = 20
max_new = 256
temp = float(cfg["trainer"].get("temperature", 0.9))
top_p = float(cfg["trainer"].get("top_p", 0.95))

records = []
for i in range(X):
    ex = ds[i]
    gen = generate_completion(ex["prompt_chat"], max_new_tokens=max_new, temperature=temp, top_p=top_p)
    records.append({
        "i": i,
        "prompt_raw": ex["prompt_raw"],
        "prompt_chat": ex["prompt_chat"],
        "chosen": ex["chosen"],
        "generated": gen,
    })

print(records[0]["generated"][:400])

KeyboardInterrupt: 

In [ ]:
from rewards import dialect_reward, cometkiwi_reward
import pandas as pd

prompts_chat = [r["prompt_chat"] for r in records]
prompts_raw  = [r["prompt_raw"] for r in records]
chosen = [r["chosen"] for r in records]
gen    = [r["generated"] for r in records]

# 1) dialect scores (chosen + generated)
dial_chosen = dialect_reward(prompts_chat, chosen)
dial_gen    = dialect_reward(prompts_chat, gen)

# 2) COMET similarity to chosen (treat chosen as reference)
# This is the key: chosen -> generated
# If your wrapper requires prompt_raw, give empty strings.
try:
    comet_chosen_gen = cometkiwi_reward(chosen, gen, prompt_raw=None, model_name="Unbabel/wmt22-cometkiwi-da", batch_size=8, force_cpu=True)
except TypeError:
    comet_chosen_gen = cometkiwi_reward(chosen, gen, prompt_raw=[""]*len(gen), model_name="Unbabel/wmt22-cometkiwi-da", batch_size=8, force_cpu=True)

for r, dc, dg, ccg in zip(records, dial_chosen, dial_gen, comet_chosen_gen):
    r["dial_chosen"] = float(dc)
    r["dial_generated"] = float(dg)
    r["dial_delta_gen_minus_chosen"] = float(dg) - float(dc)
    r["comet_chosen_to_generated"] = float(ccg)

df = pd.DataFrame(records)
df[["i","dial_chosen","dial_generated","dial_delta_gen_minus_chosen","comet_chosen_to_generated"]].head(10)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu")

emb = embedder.encode(chosen + gen, normalize_embeddings=True)
chosen_emb = emb[:len(chosen)]
gen_emb    = emb[len(chosen):]

cos = (chosen_emb * gen_emb).sum(axis=1)
df["cosine_chosen_to_generated"] = cos

df[["i","comet_chosen_to_generated","cosine_chosen_to_generated"]].head(10)

In [ ]:
# Most dialectal generated
row = df.sort_values("dial_generated", ascending=False).iloc[0]
print("PROMPT:\n", row["prompt_raw"])
print("\nCHOSEN:\n", row["chosen"])
print("\nGENERATED:\n", row["generated"])
print("\nDIALECT chosen/gen:", row["dial_chosen"], row["dial_generated"])
print("COMET(chosen->gen):", row["comet_chosen_to_generated"])
print("COS(chosen->gen):", row.get("cosine_chosen_to_generated"))

In [ ]:
metrics = [
    "dial_chosen",
    "dial_generated",
    "dial_delta",
    "comet_chosen_to_generated",
    "cosine_chosen_to_generated",
]

for m in metrics:
    if m in df.columns:
        df[m] = df[m].round(5)

In [ ]:
def inspect_rows(rows):
    for idx, row in rows.iterrows():
        print("="*80)
        print(f"INDEX: {row['i']}")
        print("-"*80)
        print("PROMPT:\n", row["prompt_raw"])
        print("\nCHOSEN:\n", row["chosen"])
        print("\nGENERATED:\n", row["generated"])
        print("\nMETRICS:")
        print(f"  Dialect chosen:    {row['dial_chosen']}")
        print(f"  Dialect generated: {row['dial_generated']}")
        print(f"  Dialect delta:     {row['dial_chosen']-row['dial_generated']}")
        print(f"  COMET(ch→gen):     {row['comet_chosen_to_generated']}")
        print(f"  Cosine(ch→gen):    {row['cosine_chosen_to_generated']}")
        print("="*80)
        print("\n\n")

top_n = df.sort_values("dial_generated", ascending=False).head(3)
inspect_rows(top_n)